# 02 — Model Experiments

Trains the four required torchvision models on the Colab T4 GPU and writes
every metric, curve, confusion matrix and checkpoint.

| Model key | torchvision model | Role |
|---|---|---|
| `VGG16` | `vgg16` | paper model |
| `ResNet18` | `resnet18` | paper model |
| `ResNet34` | `resnet34` | paper model |
| `D_ResNet` | `resnet50` | **practical approximation of the paper's D-ResNet** |

> **D-ResNet disclosure.** The paper's D-ResNet is a 50-layer bottleneck residual
> network. Its original implementation is not available, so `resnet50` from
> torchvision is used as a practical approximation. **This is not an exact
> reproduction of the original D-ResNet.** The checkpoint is named `D_ResNet_best.pth`
> only to preserve the paper's model naming.

### Rules enforced by this notebook
* No hyperparameter search, no grid/random search, no cross-validation.
* One identical training configuration for all four models.
* Model selection uses **best validation accuracy** only.
* The **test set is never touched** during training and is evaluated exactly once
  after each model finishes training.
* GPU is mandatory — training refuses to run on CPU.

In [ ]:
# ------------------------------------------------------------------
# 1. Imports / setup
# ------------------------------------------------------------------
import json
import os
import time
import random
import shutil
import warnings
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

# ------------------------------------------------------------------
# Configuration — identical for every model, no tuning performed
# ------------------------------------------------------------------
SEED = 42
CLASSES = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

IMG_SIZE = 224
EPOCHS = 60
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
# On Windows, DataLoader workers are spawned and re-import __main__, which
# deadlocks inside a Jupyter kernel. Linux/Colab is unaffected.
NUM_WORKERS = 0 if os.name == "nt" else 2
USE_AMP = True

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

SMOKE_TEST = True
SMOKE_TRAIN_IMAGES = 32
SMOKE_VAL_IMAGES = 12
SMOKE_EPOCHS = 1

MODEL_SPECS = {
    "VGG16":    {"arch": "vgg16",    "desc": "VGG16 (paper model)"},
    "ResNet18": {"arch": "resnet18", "desc": "ResNet18 (paper model)"},
    "ResNet34": {"arch": "resnet34", "desc": "ResNet34 (paper model)"},
    "D_ResNet": {"arch": "resnet50", "desc": "D-ResNet approximation = torchvision ResNet50"},
}
MODEL_ORDER = ["VGG16", "ResNet18", "ResNet34", "D_ResNet"]

PAPER_REFERENCE = {
    # Paper-reported values ONLY. Never used as our experimental results.
    "VGG16":    {"accuracy": 0.83, "precision": 0.89, "recall": 0.83},
    "ResNet18": {"accuracy": 0.78, "precision": 0.87, "recall": 0.78},
    "ResNet34": {"accuracy": 0.83, "precision": 0.89, "recall": 0.83},
    "D_ResNet": {"accuracy": 0.89, "precision": 0.90, "recall": 0.89},
}


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p] + list(p.parents):
        if (cand / "notebooks").is_dir() and (cand / "results").is_dir():
            return cand
    return p


PROJECT_ROOT = find_project_root()
RESULTS = PROJECT_ROOT / "results"

for sub in ["dataset_summary", "preprocessing", "augmentation", "metrics",
            "confusion_matrices", "training_curves", "comparisons"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)

def R(sub):
    return RESULTS / sub

print("PROJECT_ROOT :", PROJECT_ROOT)
print("torch        :", torch.__version__)
print("epochs       :", EPOCHS, "| batch:", BATCH_SIZE, "| lr:", LEARNING_RATE)
print("model order  :", MODEL_ORDER)

---
## 2. Load the processed dataset

Reads `results/preprocessing/final_split_manifest.csv` produced by notebook 01.
The manifest — not the image folders — is the single source of truth for which
sample belongs to which split, so the leakage guarantees made there carry over
here unchanged.

In [ ]:
banner_n = 0


def banner(title):
    global banner_n
    banner_n += 1
    print("\n" + "=" * 72)
    print(f"  STEP {banner_n}. {title}")
    print("=" * 72)


banner("Load processed dataset")

MANIFEST = R("preprocessing") / "final_split_manifest.csv"
if not MANIFEST.exists():
    raise SystemExit(
        f"Missing {MANIFEST}\n"
        "Run 01_Data_Preprocessing_and_EDA.ipynb first."
    )

manifest = pd.read_csv(MANIFEST)
manifest["class_idx"] = manifest["class"].map(CLASS_TO_IDX)

print(f"  manifest rows : {len(manifest)}")
print(manifest.groupby("split").size().reindex(["train", "val", "test"]).to_string())

missing_files = [p for p in manifest["sample_path"] if not Path(p).exists()]
if missing_files:
    raise SystemExit(
        f"{len(missing_files)} sample files listed in the manifest are missing, "
        f"e.g. {missing_files[:3]}. Re-run notebook 01 in this runtime."
    )
print("  all sample files present on disk")

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

# Independent leakage re-check at training time, not just trust of notebook 01.
tr_src = set(train_df["source_image_id"])
assert not (tr_src & set(val_df["source_image_id"])), "LEAKAGE train/val"
assert not (tr_src & set(test_df["source_image_id"])), "LEAKAGE train/test"
assert not (val_df["is_augmented"].any() or test_df["is_augmented"].any()), \
    "LEAKAGE augmented samples in val/test"
print("  leakage re-check PASSED (train sources disjoint from val/test; val/test un-augmented)")

---
## 3. Datasets and dataloaders

The augmentations described in the paper were already materialised to disk in
notebook 01 and are listed in the manifest. Therefore the training transform is
deliberately **deterministic** — `Resize → ToTensor → Normalize` — with no second
round of random augmentation on top.

ImageNet normalisation is an **implementation requirement** for pretrained
torchvision weights; the paper does not state its normalisation statistics.

In [ ]:
import torchvision.transforms as T

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BILINEAR),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
eval_transform = train_transform


class BSMDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["sample_path"].tolist()
        self.labels = frame["class_idx"].astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            im = im.convert("RGB")
            x = self.transform(im)
        return x, self.labels[i]


train_ds = BSMDataset(train_df, train_transform)
val_ds = BSMDataset(val_df, eval_transform)
test_ds = BSMDataset(test_df, eval_transform)

print(f"  train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                     persistent_workers=False)

def make_loader(ds, shuffle, batch_size=BATCH_SIZE):
    gen = torch.Generator()
    gen.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=False, generator=gen if shuffle else None, **loader_kwargs)

train_loader = make_loader(train_ds, True)
val_loader = make_loader(val_ds, False)
test_loader = make_loader(test_ds, False)

xb, yb = next(iter(train_loader))
print(f"  batch shape {tuple(xb.shape)} | labels {tuple(yb.shape)} | "
      f"label range [{int(yb.min())}, {int(yb.max())}] | classes {NUM_CLASSES}")
assert xb.shape[1:] == (3, IMG_SIZE, IMG_SIZE)
assert set(np.unique(yb.numpy())).issubset(set(range(NUM_CLASSES)))
print("  dataset / dataloader check PASSED")

---
## 4. GPU verification

Training is **refused** if no GPU is available — four models are never silently
trained on CPU, and mixed precision is only enabled when the device supports it.

In [ ]:
banner("GPU verification")

print("  torch.cuda.is_available() :", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("  torch.cuda.get_device_name(0) :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"  VRAM : {props.total_memory / 1024**3:.1f} GB | capability {props.major}.{props.minor}")
else:
    device = torch.device("cpu")

if device.type != "cuda":
    raise SystemExit(
        "GPU unavailable - stopping before training.\n"
        "In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU, "
        "then reconnect and re-run."
    )

print(f"\n  using device : {device}")
print(f"  mixed precision (AMP): {USE_AMP}")
print("  GPU CHECK PASSED")

---
## 5. Google Drive mount

Checkpoints and histories are mirrored to
`/content/drive/MyDrive/BSM_Mineral_Classification/models/` so that a Colab
disconnect never destroys completed work. Checkpoint binaries are **never**
committed to git.

In [ ]:
banner("Google Drive mount")

DRIVE_BASE = None
try:
    from google.colab import drive as _drive
    _drive.mount("/content/drive")
    cand = Path("/content/drive/MyDrive") / "BSM_Mineral_Classification"
    cand.mkdir(parents=True, exist_ok=True)
    (cand / "models").mkdir(parents=True, exist_ok=True)
    DRIVE_BASE = cand
    print("  Drive mounted, checkpoint dir:", cand / "models")
except Exception as e:
    print(f"  Drive unavailable ({type(e).__name__}: {e})")
    DRIVE_BASE = PROJECT_ROOT / "models"
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print("  falling back to local checkpoint dir:", DRIVE_BASE)

# In Colab DRIVE_BASE is .../BSM_Mineral_Classification so checkpoints land in
# .../BSM_Mineral_Classification/models/. The local fallback already IS the
# models directory, so do not append a second "models" segment.
CKPT_DIR = DRIVE_BASE if DRIVE_BASE.name == "models" else DRIVE_BASE / "models"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("  CKPT_DIR =", CKPT_DIR)

---
## 6. Models

All four architectures come straight from `torchvision` with ImageNet weights —
none is hand-implemented. Only the final classifier layer is replaced with a
6-output linear layer.

In [ ]:
banner("Model construction")

from torchvision import models as tv_models


def build_model(key):
    arch = MODEL_SPECS[key]["arch"]
    weights = "IMAGENET1K_V1"
    if arch == "vgg16":
        net = tv_models.vgg16(weights=weights)
        net.classifier[6] = nn.Linear(net.classifier[6].in_features, NUM_CLASSES)
    else:
        net = getattr(tv_models, arch)(weights=weights)
        net.fc = nn.Linear(net.fc.in_features, NUM_CLASSES)
    return net.to(device)


def param_count(net):
    return sum(p.numel() for p in net.parameters() if p.requires_grad)


for key in MODEL_ORDER:
    net = build_model(key)
    n = param_count(net)
    out = net.fc.out_features if hasattr(net, "fc") else net.classifier[6].out_features
    print(f"  {key:<10} arch={MODEL_SPECS[key]['arch']:<9} "
          f"trainable params={n/1e6:6.2f} M | outputs={out}")
    assert out == NUM_CLASSES
    del net
print("  model construction PASSED")

In [ ]:
# ------------------------------------------------------------------
# Metrics: accuracy, precision, recall (=sensitivity), F1, specificity
# Specificity is one-vs-rest: TN / (TN + FP)
# ------------------------------------------------------------------
def compute_all_metrics(y_true, y_pred, classes=CLASSES):
    labels = list(range(len(classes)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)

    total = cm.sum()
    specificity = {}
    for i, cls in enumerate(classes):
        tp = int(cm[i, i])
        fn = int(cm[i].sum()) - tp
        fp = int(cm[:, i].sum()) - tp
        tn = total - tp - fn - fp
        specificity[cls] = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    spec_arr = np.array([specificity[c] for c in classes])

    rows = []
    for i, cls in enumerate(classes):
        rows.append({
            "class": cls, "class_idx": i,
            "precision": float(prec[i]), "recall": float(rec[i]),
            "sensitivity": float(rec[i]), "f1_score": float(f1[i]),
            "specificity": float(spec_arr[i]), "support": int(sup[i]),
        })
    rows.append({
        "class": "macro avg", "class_idx": -1,
        "precision": float(prec.mean()), "recall": float(rec.mean()),
        "sensitivity": float(rec.mean()), "f1_score": float(f1.mean()),
        "specificity": float(spec_arr.mean()), "support": int(sup.sum()),
    })
    w = sup / max(sup.sum(), 1)
    rows.append({
        "class": "weighted avg", "class_idx": -2,
        "precision": float((prec * w).sum()), "recall": float((rec * w).sum()),
        "sensitivity": float((rec * w).sum()), "f1_score": float((f1 * w).sum()),
        "specificity": float((spec_arr * w).sum()), "support": int(sup.sum()),
    })
    metrics_df = pd.DataFrame(rows)
    metrics_df.loc[len(metrics_df)] = {
        "class": "accuracy", "class_idx": -3, "precision": float(acc),
        "recall": float(acc), "sensitivity": float(acc), "f1_score": float(acc),
        "specificity": np.nan, "support": int(len(y_true)),
    }
    return float(acc), metrics_df, cm

---
## 7. Smoke test

`SMOKE_TEST = True` runs one epoch over a tiny subset with the fastest model and
exercises every part of the pipeline: dataset loading, transforms, dataloader,
GPU, forward pass, backward pass, optimizer step, validation, metrics, confusion
matrix, plotting and checkpoint saving.

Real training begins only after `SMOKE TEST PASSED` is printed.

In [ ]:
banner("Training / evaluation functions")


def run_epoch(model, loader, criterion, optimizer=None, scaler=None, use_amp=True):
    training = optimizer is not None
    model.train() if training else model.eval()
    # autocast is applied to the training pass only. Evaluation runs in fp32:
    # fp16 forward passes through VGG16's deep stack can overflow to inf/nan and
    # corrupt the validation loss, which is the metric used for checkpointing.
    amp_enabled = bool(use_amp and training)

    total_loss, correct, seen = 0.0, 0, 0
    all_true, all_pred = [], []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                outputs = model(images)
                loss = criterion(outputs, targets)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        else:
            with torch.no_grad():
                outputs = model(images)
                loss = criterion(outputs, targets)

        total_loss += loss.item() * targets.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        seen += targets.size(0)
        all_true.append(targets.detach().cpu().numpy())
        all_pred.append(outputs.detach().argmax(1).cpu().numpy())

    return (total_loss / max(seen, 1), correct / max(seen, 1),
            np.concatenate(all_true) if all_true else np.array([]),
            np.concatenate(all_pred) if all_pred else np.array([]))

In [ ]:
def plot_confusion_matrix(cm, classes, title, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cm_norm = cm.astype("float") / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.6))
    for axi, mat, fmt, name in [(ax[0], cm, "d", "counts"), (ax[1], cm_norm, ".2f", "row-normalised")]:
        im = axi.imshow(mat, cmap="Blues")
        fig.colorbar(im, ax=axi, fraction=0.046)
        axi.set_xticks(range(len(classes))); axi.set_xticklabels(classes, rotation=30, ha="right")
        axi.set_yticks(range(len(classes))); axi.set_yticklabels(classes)
        axi.set_xlabel("predicted"); axi.set_ylabel("true"); axi.set_title(f"confusion matrix ({name})")
        thr = mat.max() / 2.0 if mat.max() > 0 else 0.5
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                axi.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                         color="white" if mat[i, j] > thr else "black", fontsize=9)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.savefig(path, dpi=140); plt.close()


def plot_curves(history, title, model_key):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ep = history["epoch"]
    plt.figure(figsize=(7.5, 4.6))
    plt.plot(ep, history["train_loss"], label="train loss", color="#C44E52")
    plt.plot(ep, history["val_loss"], label="validation loss", color="#4C72B0")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(f"{model_key} — loss")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(R("training_curves") / f"{model_key}_loss.png", dpi=140); plt.close()

    plt.figure(figsize=(7.5, 4.6))
    plt.plot(ep, history["train_accuracy"], label="train accuracy", color="#C44E52")
    plt.plot(ep, history["val_accuracy"], label="validation accuracy", color="#4C72B0")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title(f"{model_key} — accuracy")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(R("training_curves") / f"{model_key}_accuracy.png", dpi=140); plt.close()
    plt.close("all")


def save_checkpoint(net, model_key, epoch, val_acc, extra=None):
    path = CKPT_DIR / f"{model_key}_best.pth"
    spec = MODEL_SPECS.get(model_key, {})
    payload = {
        "model_key": model_key,
        "architecture": spec.get("arch", model_key),
        "state_dict": net.state_dict(),
        "epoch": epoch,
        "val_accuracy": val_acc,
        "classes": CLASSES,
        "img_size": IMG_SIZE,
        "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
        "seed": SEED,
        "note": ("D_ResNet is a torchvision ResNet50 practical approximation of the "
                 "paper's D-ResNet, not the original implementation."),
    }
    if extra:
        payload.update(extra)
    tmp = path.with_suffix(".pth.tmp")
    torch.save(payload, tmp)
    tmp.replace(path)
    meta = CKPT_DIR / f"{model_key}_best_meta.json"
    meta.write_text(json.dumps({k: v for k, v in payload.items() if k != "state_dict"}, indent=2))
    return path

In [ ]:
banner("Smoke test")

if SMOKE_TEST:
    set_seed(SEED)
    smoke_train = torch.utils.data.Subset(train_ds, list(range(min(SMOKE_TRAIN_IMAGES, len(train_ds)))))
    smoke_val = torch.utils.data.Subset(val_ds, list(range(min(SMOKE_VAL_IMAGES, len(val_ds)))))
    smoke_train_loader = make_loader(smoke_train, True, batch_size=8)
    smoke_val_loader = make_loader(smoke_val, False, batch_size=8)

    net = build_model("ResNet18")
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None

    t0 = time.time()
    for epoch in range(1, SMOKE_EPOCHS + 1):
        trl, tra, _, _ = run_epoch(net, smoke_train_loader, criterion, optimizer, scaler, USE_AMP)
        val, acc, yv, pv = run_epoch(net, smoke_val_loader, criterion, None, None, USE_AMP)
        print(f"  epoch {epoch}/{SMOKE_EPOCHS} | train loss {trl:.4f} acc {tra:.4f} "
              f"| val loss {val:.4f} acc {acc:.4f}")

    m_acc, m_df, m_cm = compute_all_metrics(yv, pv)
    print("\n  smoke metrics:")
    print(m_df.to_string(index=False))

    pd.DataFrame(m_cm, index=CLASSES, columns=CLASSES).to_csv(
        R("confusion_matrices") / "SMOKE_TEST_confusion_matrix.csv")
    plot_confusion_matrix(m_cm, CLASSES, "SMoke test - ResNet18",
                          R("confusion_matrices") / "SMOKE_TEST_confusion_matrix.png")
    plot_curves(pd.DataFrame([{"epoch": 1, "train_loss": trl, "val_loss": val,
                               "train_accuracy": tra, "val_accuracy": acc}]),
                "smoke", "SMOKE_TEST")
    ck = save_checkpoint(net, "SMOKE_TEST", SMOKE_EPOCHS, acc)
    print(f"\n  checkpoint written: {ck.name} ({ck.stat().st_size/1e6:.1f} MB)")

    del net, optimizer
    torch.cuda.empty_cache()

    # every required artefact exists?
    required = [
        R("confusion_matrices") / "SMOKE_TEST_confusion_matrix.png",
        R("training_curves") / "SMOKE_TEST_loss.png",
        R("training_curves") / "SMOKE_TEST_accuracy.png",
        CKPT_DIR / "SMOKE_TEST_best.pth",
    ]
    for p in required:
        assert p.exists(), f"smoke test artefact missing: {p}"
        print(f"  ok  {p.name}")

    print(f"\n  smoke test wall time: {time.time()-t0:.1f}s")
    print("\n" + "=" * 72)
    print("SMOKE TEST PASSED")
    print("=" * 72 + "\n")
else:
    print("  SMOKE_TEST disabled")

---
## 8. Full training — 4 models x 60 epochs

Identical configuration for all four models. `SMOKE_TEST` is now `False`.

Each model, on completion, immediately writes its history CSV, classification
report, confusion matrix (PNG + CSV), training curves and best checkpoint to
Google Drive — so an interrupted Colab session never loses completed work.
A model whose outputs already exist is **skipped, not restarted**.

In [ ]:
SMOKE_TEST = False

def already_done(model_key):
    hist = R("metrics") / f"{model_key}_history.csv"
    ckpt = CKPT_DIR / f"{model_key}_best.pth"
    return hist.exists() and ckpt.exists()


def train_model(model_key, epochs=EPOCHS):
    set_seed(SEED)
    print(f"\n{'-'*72}\n  TRAINING {model_key}  ({MODEL_SPECS[model_key]['desc']})\n{'-'*72}")

    net = build_model(model_key)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None

    history, best_val_acc, best_epoch = [], -1.0, -1
    train_start = time.time()

    for epoch in range(1, epochs + 1):
        ep_start = time.time()
        trl, tra, _, _ = run_epoch(net, train_loader, criterion, optimizer, scaler, USE_AMP)
        val, vacc, _, _ = run_epoch(net, val_loader, criterion, None, None, USE_AMP)
        ep_time = time.time() - ep_start

        # A non-finite validation loss means the forward pass overflowed; refuse
        # to promote such an epoch to "best" so a broken state cannot be saved.
        val_finite = bool(np.isfinite(val))
        if not val_finite:
            print(f"  WARNING epoch {epoch}: non-finite validation loss ({val}); "
                  "not eligible for best-checkpoint selection")
            vacc = -1.0

        history.append({
            "epoch": epoch,
            "train_loss": trl, "val_loss": val,
            "train_accuracy": tra, "val_accuracy": vacc,
            "epoch_time_seconds": ep_time,
        })
        if vacc > best_val_acc:
            best_val_acc, best_epoch = vacc, epoch
            save_checkpoint(net, model_key, epoch, vacc)

        if epoch % 5 == 0 or epoch == 1 or epoch == epochs:
            print(f"  epoch {epoch:>3}/{epochs} | train loss {trl:.4f} acc {tra:.4f} "
                  f"| val loss {val:.4f} acc {vacc:.4f} | {ep_time:5.1f}s"
                  f"{'  <- best' if epoch == best_epoch else ''}")

    total_train_time = time.time() - train_start

    # incremental save — history first, so an interruption still leaves a record
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(R("metrics") / f"{model_key}_history.csv", index=False)
    (CKPT_DIR / f"{model_key}_history.csv").write_text(hist_df.to_csv(index=False))

    # ---- test set touched exactly once, after training is complete -------
    test_loss, _, y_true, y_pred = run_epoch(net, test_loader, criterion, None, None, USE_AMP)
    test_acc, metrics_df, cm = compute_all_metrics(y_true, y_pred)

    print(f"\n  TEST (evaluated once, after training): accuracy {test_acc:.4f}")
    print(metrics_df.to_string(index=False))

    # ---- artefacts ------------------------------------------------------
    rep = metrics_df.copy()
    rep.insert(0, "model", model_key)
    rep.to_csv(R("metrics") / f"{model_key}_classification_report.csv", index=False)
    (CKPT_DIR / f"{model_key}_classification_report.csv").write_text(rep.to_csv(index=False))

    cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
    cm_df.to_csv(R("confusion_matrices") / f"{model_key}_confusion_matrix.csv")
    plot_confusion_matrix(cm, CLASSES, f"{model_key} (ResNet/torchvision {MODEL_SPECS[model_key]['arch']}) "
                                      f"- test confusion matrix",
                          R("confusion_matrices") / f"{model_key}_confusion_matrix.png")

    plot_curves(hist_df, f"{model_key} training curves", model_key)

    macro = metrics_df[metrics_df["class"] == "macro avg"].iloc[0]
    wtd = metrics_df[metrics_df["class"] == "weighted avg"].iloc[0]

    summary = {
        "model": model_key,
        "architecture": MODEL_SPECS[model_key]["arch"],
        "description": MODEL_SPECS[model_key]["desc"],
        "is_dresnet_approximation": model_key == "D_ResNet",
        "epochs": epochs,
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "loss": "CrossEntropyLoss",
        "seed": SEED,
        "mixed_precision": bool(USE_AMP),
        "train_samples": len(train_ds),
        "val_samples": len(val_ds),
        "test_samples": len(test_ds),
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_acc,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_precision_macro": macro["precision"],
        "test_recall_macro": macro["recall"],
        "test_f1_macro": macro["f1_score"],
        "test_specificity_macro": macro["specificity"],
        "test_precision_weighted": wtd["precision"],
        "test_recall_weighted": wtd["recall"],
        "test_f1_weighted": wtd["f1_score"],
        "test_specificity_weighted": wtd["specificity"],
        "total_training_seconds": total_train_time,
        "mean_epoch_seconds": hist_df["epoch_time_seconds"].mean(),
        "checkpoint_path": str(CKPT_DIR / f"{model_key}_best.pth"),
        "completed_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    }
    (CKPT_DIR / f"{model_key}_summary.json").write_text(json.dumps(summary, indent=2))
    print(f"  artefacts written; total training time {total_train_time/60:.1f} min")

    del net, optimizer, scaler
    torch.cuda.empty_cache()
    return summary


summaries = []
for key in MODEL_ORDER:
    if already_done(key):
        p = CKPT_DIR / f"{key}_summary.json"
        if p.exists():
            s = json.loads(p.read_text())
        else:
            s = {"model": key, "architecture": MODEL_SPECS[key]["arch"],
                 "description": MODEL_SPECS[key]["desc"],
                 "is_dresnet_approximation": key == "D_ResNet",
                 "note": "outputs already present; skipped to preserve prior results"}
        print(f"\n  {key}: outputs already exist, SKIPPING (not restarting)")
        summaries.append(s)
        continue
    summaries.append(train_model(key))

In [ ]:
banner("Aggregate results")

final_results = pd.DataFrame([s for s in summaries if "test_accuracy" in s])
if final_results.empty:
    raise SystemExit("No completed training runs found.")
final_results = final_results.sort_values("test_accuracy", ascending=False)
final_results.to_csv(R("metrics") / "final_results.csv", index=False)

display_cols = ["model", "architecture", "best_epoch", "best_val_accuracy", "test_accuracy",
                "test_precision_macro", "test_recall_macro", "test_f1_macro",
                "test_specificity_macro", "total_training_seconds"]
print(final_results[display_cols].round(4).to_string(index=False))

tt = final_results[["model", "architecture", "epochs", "batch_size",
                    "total_training_seconds", "mean_epoch_seconds",
                    "best_epoch", "best_val_accuracy"]].copy()
tt["total_training_minutes"] = tt["total_training_seconds"] / 60.0
tt.to_csv(R("metrics") / "training_time.csv", index=False)
print("\n" + tt.round(2).to_string(index=False))

# Mirror lightweight results to Drive so they survive a disconnect.
for name in ["final_results.csv", "training_time.csv"]:
    src = R("metrics") / name
    if src.exists():
        shutil.copy2(src, CKPT_DIR / name)
print(f"\n  results mirrored to {CKPT_DIR}")

paper = pd.DataFrame([
    {"model": k, "paper_accuracy": v["accuracy"], "paper_precision": v["precision"],
     "paper_recall": v["recall"]}
    for k, v in PAPER_REFERENCE.items()
])
paper.to_csv(R("metrics") / "paper_reference_values.csv", index=False)
print("\n  paper reference values saved for notebook 03 "
      "(reference targets only, NOT our results)")

---
## Notebook 02 complete

| Artefact | Path |
|---|---|
| per-epoch history | `results/metrics/<Model>_history.csv` |
| classification report | `results/metrics/<Model>_classification_report.csv` |
| aggregated results | `results/metrics/final_results.csv` |
| training time | `results/metrics/training_time.csv` |
| confusion matrix PNG | `results/confusion_matrices/<Model>_confusion_matrix.png` |
| confusion matrix CSV | `results/confusion_matrices/<Model>_confusion_matrix.csv` |
| loss curve | `results/training_curves/<Model>_loss.png` |
| accuracy curve | `results/training_curves/<Model>_accuracy.png` |
| best checkpoints | `<Drive>/BSM_Mineral_Classification/models/<Model>_best.pth` |

Next: **`03_Final_Comparison.ipynb`** (reads these CSVs only — it does not retrain).